# **Sequence to Sequence RNN Models: Language Translation**

## Objectives


 - Comprehend recurrent neural networks (RNN) architecture
 - Create an Encoder-Decoder model for a translation task
 - Train and evaluate the model
 - Create a generator for the translation task
 - Explain concepts related to Perplexity and BLEU score and use them for evaluating translations


# Background

Sequence-to-sequence (Seq2seq) models have transformed the landscape of natural language processing (NLP), enabling breakthroughs in tasks such as speech recognition, dialogue systems, and code generation. These models leverage Recurrent Neural Networks (RNNs) to map variable-length input sequences to variable-length output sequences through an encoder-decoder architecture.

## History of Sequence-to-Sequence Models

Sequence-to-sequence models emerged from the limitations of fixed-size input/output neural networks.
Researchers identified the necessity for architectures capable of processing arbitrary-length sequences, particularly for language translation tasks.
The landmark contribution of Cho et al. (2014) established the encoder-decoder framework that forms the foundation of modern seq2seq models.

Here are some main applications of seq2seq models:
- **Speech Recognition**: Converting spoken audio sequences into written text sequences.
- **Code Generation**: Producing executable code from natural language descriptions.
- **Dialogue Systems**: Generating contextually appropriate responses in conversational AI.

And many more tasks that require mapping one sequence domain to another.

## Introduction to RNNs

RNNs are a specialized class of neural networks built to handle time-dependent and sequential data.
They maintain a hidden state ($h_t$) that acts as a compressed memory of all previously seen inputs.
RNNs utilize feedback connections that propagate information across time steps during processing.

Recurrent Neural Networks (RNNs) operate on sequences and utilize previous states to influence the current state. Here's the general formulation of a simple RNN:

**Given:**
- $\mathbf{x}_t$: input vector at time step $t$
- $\mathbf{h}_{t-1}$: hidden state vector from the previous time step
- $\mathbf{W}_x$ and $\mathbf{W}_h$: weight matrices for the input and hidden state, respectively
- $\mathbf{b}$: bias vector
- $\sigma$: activation function (often a sigmoid or tanh)

The update equation for the hidden state $\mathbf{h}_t$ is as follows:

$$
\mathbf{h}_t = \sigma(\mathbf{W}_x \cdot \mathbf{x}_t + \mathbf{W}_h \cdot \mathbf{h}_{t-1} + \mathbf{b})
$$

It can be observed that the hidden state at time $t$ is jointly determined by the current input and the previous hidden state, enabling the network to accumulate contextual information over time.

For the output (if you're making a prediction at each time step):

$$
\mathbf{y}_t = \text{softmax}(\mathbf{W}_o \cdot \mathbf{h}_t + \mathbf{b}_o)
$$

**Where:**
- $\mathbf{W}_o$: weight matrix for the output
- $\mathbf{b}_o$: bias vector for the output

Depending on the specific task, an RNN cell can either emit an output from $h_t$ or pass it forward exclusively as internal memory. To better understand how this memory mechanism operates in practice, consider the following state transition diagram:

![State Transition Diagram](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Screenshot%202023-10-19%20at%2011.29.23%E2%80%AFAM.png)

The diagram illustrates a state transition system with three distinct states, represented by the prominent purple circles. Each state is uniquely identified by a value of $h$: $h = -1$, $h = 0$, and $h = 1$.

### State $h = -1$
- Remains in the same state when $x = 1$ (shown by the yellow self-loop).
- Transitions to the $h = 0$ state upon receiving input $x = -1$ (indicated by the red arrow).

### State $h = 0$
- Shifts to the $h = -1$ state when input $x = 1$ is received (indicated by the red arrow).
- Advances to the $h = 1$ state when input $x = -1$ is received (indicated by the red arrow).

### State $h = 1$
- Remains in the same state when $x = -1$ (shown by the yellow self-loop).
- Transitions back to the $h = 0$ state upon receiving input $x = 1$ (indicated by the red arrow).

---

In summary, this diagram effectively captures the dynamic behavior of an RNN's hidden state, demonstrating how the network transitions between memory states based on sequential inputs $x$. Depending on both the current state and the incoming input, the system either shifts to a new state or holds its current configuration — mirroring how an RNN retains and updates its internal memory over time.


# LSTM and Sequence-to-Sequence Architecture

## Long Short-Term Memory (LSTM)

In real-world applications, standard RNNs are rarely used in their basic form. Instead, advanced variants such as **Long Short-Term Memory (LSTM)** and **Gated Recurrent Units (GRU)** are preferred, as they effectively overcome the vanishing gradient problem that limits basic RNNs.

An LSTM cell is built around three core mechanisms known as **gates**:

- The **Input Gate** regulates the flow of new information into the cell's memory. By examining both the current input and the prior hidden state, it selectively determines which portions of the incoming data are worth retaining.

- The **Forget Gate** is responsible for clearing out outdated or irrelevant information from memory. It evaluates the current input alongside the previous hidden state to decide which stored information is no longer useful and should be discarded.

- The **Output Gate** governs what portion of the cell's memory is passed forward as output. Based on the current input and previous hidden state, it filters the memory to produce a meaningful output signal.

The fundamental strength of LSTMs lies in their dedicated memory state, which can independently retain or release information as needed. This design enables LSTMs to effectively capture long-range dependencies and preserve critical context from earlier positions in a sequence.

---

## Sequence-to-Sequence Architecture

Seq2seq models are structured around an **Encoder-Decoder** framework. The encoder processes the entire input sequence and compresses it into a compact representation known as the **context vector** ($h_t$). The decoder then takes this context vector and uses it to generate the output sequence step by step.

### How It Works — Translation Example

Consider translating the English sentence *"I love to travel"* into French *"J'adore voyager"* — a classic seq2seq task.

**Encoder Phase:**
- Words from the input sentence are fed into the encoder one at a time.
- At each step, an RNN cell takes the current word ($x_t$) and its internal memory ($h_t$), processes them together, and passes an updated context vector ($h_{t+1}$) to the next cell.
- Once the entire input sequence is consumed, the final context vector captures a compressed understanding of the whole sentence.

**Decoder Phase:**
- The context vector is handed off to the decoder.
- The decoder consists of RNN cells that generate output words one at a time.
- At each step, a decoder cell receives the previously generated word along with the updated context vector from the prior cell, and produces the next output word ($y_t$).

This encoder-decoder design allows seq2seq models to generate output sequences of **arbitrary length**, making them highly flexible for tasks like translation, summarization, and dialogue generation.


# Encoder Implementation in PyTorch

## Overview

To build the encoder in PyTorch, you subclass `torch.nn.Module` and implement two essential methods: `__init__()` for defining the architecture and `forward()` for specifying the data flow.

---

## Constructor Parameters — `__init__()`

The following parameters configure the encoder's structure:

- **`vocab_len`** — Represents the total number of unique tokens in the vocabulary. This value is derived after preprocessing the dataset and serves as the dimensionality of the model's input.

- **`embedding_dim`** — Defines the size of the output embedding vector. For a small-scale demo application, values in the range of **256 to 512** are generally recommended.

- **`n_layers`** — Specifies how many LSTM layers to stack. While a single layer is used in the initial setup, this parameter is included upfront to allow easy scaling in future iterations.

- **`hid_dim`** — Controls the dimensionality of both the hidden state and the cell state within the LSTM.

- **`dropout`** — A regularization parameter that randomly deactivates a fraction of neurons during training to help prevent overfitting.

---

## Layer Definitions

Two core layers are defined inside `__init__()`:

- **Embedding Layer** — Maps raw input tokens to dense vector representations. Its input dimension is set to `vocab_len` and its output dimension to `embedding_dim`.

- **LSTM Layer** — Accepts the embedding vectors (of size `embedding_dim`) as input and produces three outputs: `output`, `hidden`, and `cell`. The internal capacity of the LSTM is controlled by `hid_dim`.

---

## Forward Pass — `forward()`

The data flows through the encoder as follows:

1. The **Embedding Layer** receives the input batch and internally maps each token to a one-hot representation before converting it into a dense embedding vector.
2. The **LSTM Layer** then processes these embeddings and produces three vectors: `output`, `hidden`, and `cell`.
3. Since the encoder's sole responsibility is to compress the input into a **context vector**, the `output` is discarded. Only `hidden` and `cell` are returned and passed on to the decoder.

---

> **Note:** When using an LSTM, the context vector consists of both the **hidden state** and the **cell state**. In contrast, if a GRU were used instead, only the **hidden state** would be available, as GRUs do not maintain a separate cell state.


### Import Libraries

In [2]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### **Define Encoder Class**

In [7]:
class Encoder(nn.Module):
    
    def __init__(self, vocab_len, embed_dim, hid_dim, n_layers, dropout_prob ):
        super().__init__()
        
        # define class attributes
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embed_dim = embded_dim
        self.vocab_len = vocab_len
        
        #define embedding layer for inputs
        self.embedding = nn.Embedding(self.vocab_len, self.embed_dim)
        #define LSTM layer
        self.lstm = nn.LSTM(embed_dim,hid_dim,n_layers,dropout = dropout_prob)
        # define drop out for randomly drop neurons
        self.dropout = nn.Dropout(dropout_prob)
        
    def forward(self, input_batch):
        #input_batch = [src len, batch size]
        embed_out = self.dropout(self.embedding(input_batch))
        embed_out = embed_out.to(device)
        
        #outputs = [src len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]
        
        output, (hidden,cell) = self.lstm(embed_out)
        
        return hidden, cell
        

Test the Encoder Class

In [9]:
vocab_size = 10
hid_dim  = 8
embded_dim = 10
n_layers = 1
dropout_prob = 0.5

encoder_t = Encoder(vocab_size,embded_dim,hid_dim,n_layers,dropout_prob)

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [10]:
src_data = torch.tensor([[0,3,4,2,1]])
src_data.shape

torch.Size([1, 5])

In [11]:
# so LSTM must be feed src len firstly. so we need to transpose the input   
src_data_t = src_data.t().to(device)
src_data_t.shape

torch.Size([5, 1])

In [12]:
hidden_t, cell_t = encoder_t(src_data_t)
print("Hidden tensor from encoder:",hidden_t ,"\nCell tensor from encoder:", cell_t)

Hidden tensor from encoder: tensor([[[ 0.0342,  0.0755,  0.2131,  0.2480, -0.1339, -0.2807,  0.0067,
           0.0554]]], grad_fn=<StackBackward0>) 
Cell tensor from encoder: tensor([[[ 0.0831,  0.0890,  0.9846,  0.3949, -0.2722, -0.4296,  0.0090,
           0.1916]]], grad_fn=<StackBackward0>)


The encoder receives the entire source sequence as input, which is a sequence of words or tokens. This sequence is first converted into embeddings and then processed step-by-step by the LSTM. At each time step, the LSTM updates its hidden and cell states, which together store information about both short-term and long-term dependencies in the sequence.

As the LSTM processes the input, its hidden states act as a form of memory that captures the contextual meaning of the sequence over time. After the entire sequence has been processed, the final hidden state (along with the final cell state) represents a compressed summary of the input. This summary is often referred to as the context vector, as it encodes the overall meaning of the source sequence and is passed to the decoder.

### **Define Decoder Class**

The decoder class inherits from nn.Module, which is a base class for all neural network modules in PyTorch.
The constructor (__init__ method) initializes the parameters and layers of the decoder.
- `output_dim` is the number of possible output values(target vocab length).
- `emb_dim` is the dimensionality of the embedding layer.
- `hid_dim` is the dimensionality of the hidden state in the LSTM.
- `n_layers` is the number of layers in the LSTM.
- `dropout` is the dropout probability.

The decoder contains the following layers:
- `embedding`: An embedding layer that maps the output values to dense vectors of size emb_dim.
- `lstm`: An LSTM layer that takes the embedded input and produces hidden states of size hid_dim.
-  `fc_out`: A linear layer that maps the LSTM output to the output dimension output_dim.
- `softmax`: A log-softmax activation function applied to the output to obtain a probability distribution over the output values.
- `dropout`: A dropout layer that applies dropout to the embedded input.


In [13]:
class Decoder (nn.Module):
    
    def __init__(self, output_dim, embed_dim, hid_dim, n_layers, dropout):
        super().__init__()
        
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.output_dim = output_dim
        
        #define embedded layer
        self.embedding = nn.Embedding(output_dim,embed_dim)
        #define lstm layer
        self.lstm = nn.LSTM(embed_dim,hid_dim,n_layers,dropout = dropout)
        # define output as fully connected network
        self.fc_out = nn.Linear(in_features=hid_dim, out_features=output_dim)
        #define softmax layer
        self.softmax = nn.LogSoftmax(dim=1)
        #define dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        
        #input = [batch size]

        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #n directions in the decoder will both always be 1, therefore:
        
        #hidden = [n layers, batch size, hid dim]
        #context = [n layers, batch size, hid dim]
        input = input.unsqueeze(0)
        #input = [1, batch size]
        
        embedded = self.dropout(self.embedding(input))
        #embedded = [1, batch size, emb dim]
        
        output, (hidden_out,cell_out)  = self.lstm(embedded,(hidden,cell))
        #output = [seq len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]
        
        #seq len and n directions will always be 1 in the decoder, therefore:
        #output = [1, batch size, hid dim]
        #hidden = [n layers, batch size, hid dim]
        #cell = [n layers, batch size, hid dim]
        
        prediction = self.fc_out(output.squeeze(0))
        
        prediction_prob = self.softmax(prediction)
        #prediction_prob = [batch size, output dim]
        
        return prediction_prob, hidden_out, cell_out

Test the Decoder

In [18]:
output_dim = 6
embed_dim = 10
hid_dim = 8
n_layers = 1
dropout_prob=0.5

decoder_t = Decoder(output_dim,embed_dim,hid_dim,n_layers,dropout_prob)

In [19]:
input_t = torch.tensor([0]).to(device) #<bos>
input_t.shape

torch.Size([1])

In [20]:
prediction_prob, hidden_decoder, cell_decorder = decoder_t(input_t,hidden_t,cell_t)
print("Prediction:", prediction_prob, '\nHidden:',hidden_decoder,'\nCell:', cell_decorder)

Prediction: tensor([[-1.6769, -1.8907, -1.7155, -1.8800, -1.6508, -1.9824]],
       grad_fn=<LogSoftmaxBackward0>) 
Hidden: tensor([[[ 0.2865,  0.0602,  0.2744,  0.1907, -0.0750, -0.0945, -0.4320,
           0.1557]]], grad_fn=<StackBackward0>) 
Cell: tensor([[[ 0.4834,  0.1786,  1.1731,  0.7914, -0.4557, -0.2957, -0.6826,
           0.4953]]], grad_fn=<StackBackward0>)
